# Kapitel 4 – Övningsuppgifter: Klassificering

Det här är mina svar på övningsfrågorna till kapitel 4, baserade på boken *"Lär dig AI från grunden - Tillämpad maskininlärning med Python"* (Prgomet, Johnson, Solberg, Rundberg Streuli) och övningsuppgifterna från bokens GitHub-repo.

## Fråga 1 – Vad kännetecknar klassificeringsproblem? Ge några exempel på tillämpningsområden.

Klassificering handlar om att förutsäga en kategorisk utdata (y) utifrån indata (x), där y kan anta två eller flera klasser. Skillnaden mot regression är egentligen ganska enkel att komma ihåg: i regression är utdatan kontinuerlig, medan den i klassificering är diskret och begränsad till ett visst antal klasser.

Man brukar dela in det i tre varianter. Binär klassificering är den enklaste formen, där det bara finns två möjliga klasser, till exempel {churna, stanna} eller {1, 0}. Multiklass klassificering är när utdatan kan anta fler än två klasser, som {hund, katt, häst, fågel}. Sen finns det multioutput klassificering, där man predicerar flera utdata-variabler samtidigt – till exempel om en person är över 18 år *och* vilket kön personen har. Om någon av de outputsen dessutom kan ha fler än två klasser kallas det multioutput multiklass klassificering. Boken själv håller sig till binär och multiklass klassificering, så det är de två man behöver kunna bäst.

När en modell tränas lär den sig en **beslutsgräns** (decision boundary), det vill säga en gräns som separerar klasserna i rummet som spänns upp av indata-variablerna (Figur 4.1). Den kan vara linjär, som hos logistisk regression, eller olinjär, som hos random forest.

Boken tar upp flera exempel på tillämpningsområden i avsnitt 4.1. Inom kundanalys kan man predicera om en kund kommer churna baserat på till exempel antal köp, ålder och antal supportärenden. Inom sjukvård kan man förutsäga om en patient är sjuk eller frisk utifrån medicinsk data som blodtryck och puls. Bildigenkänning handlar om att identifiera objekt i bilder – färg, form, textur – exempelvis skadedjur i jordbruksmiljöer, medan anomalidetektion används för att hitta defekta produkter i tillverkningsprocesser med hjälp av kamerabilder. Inom finans kan man försöka predicera om en aktie kommer stiga i värde, och inom kreditrisk bedöms om en bankkund ska beviljas ett huslån baserat på inkomst, ålder, civilstånd och liknande.

## Fråga 2 – Förklara hur OvR- och OvO-algoritmerna fungerar.

Binära klassificeringsmodeller kan med hjälp av **OvR** (One-vs-Rest) och **OvO** (One-vs-One) byggas ut till att klara multiklass klassificering (avsnitt 4.1.1).

Tanken med OvR är att man tränar en egen binär modell för varje klass i problemet, en modell som bara avgör om en datapunkt tillhör just den klassen eller inte. Ska man till exempel klassificera siffrorna 0–9 tränar OvR alltså tio klassificerare: en som avgör "är det en 0:a eller inte?", en som avgör "är det en 1:a eller inte?" och så vidare. Vid en ny prediktion körs samtliga modeller, och den som ger högst score (till exempel högst sannolikhet) vinner.

OvO gör det annorlunda – där tränas istället en modell för varje par av klasser. Om det finns N klasser blir det $(N \times (N-1))/2$ modeller, så för siffrorna 0–9 tränas en modell för paret {0,1}, en för {0,2}, och så vidare ända till {8,9}. Varje modell predikterar en av de två klasserna i sitt par, och när en ny observation ska klassificeras körs alla modeller och den klass som får flest röster totalt blir den slutgiltiga prediktionen.

Det som är skönt är att man sällan behöver implementera det här själv – kör man en binär modell på multiklass-data väljer scikit-learn automatiskt OvO eller OvR åt en.

## Fråga 3 – Förklara följande utvärderingsmått: a) Confusion matrix, b) Accuracy, c) Precision, d) Recall, e) F1-score, f) ROC-kurvan.

**a) Confusion matrix**

En confusion matrix är egentligen bara en matris som visar hur väl en modell presterar genom att ställa sanna värden mot predikterade värden (avsnitt 4.2.1, Figur 4.2–4.3). Varje rad brukar representera den sanna klassen och varje kolumn den predikterade klassen (eller tvärtom, beroende på hur man vänder på den). Diagonalen, från övre vänstra hörnet till nedre högra, visar alla punkter som klassificerats rätt. Det man främst tittar på i matrisen är fyra saker: TP (True Positive, modellen gissade rätt på en positiv klass), TN (True Negative, rätt på en negativ klass), FP (False Positive, modellen sa positiv men hade fel – typ I-fel) och FN (False Negative, modellen sa negativ men hade fel – typ II-fel). Dessa fyra siffror ligger sedan till grund för alla måtten nedan.

**b) Accuracy**

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

Accuracy är helt enkelt andelen av alla observationer som klassificerades rätt. Problemet är att ett högt värde inte automatiskt betyder att modellen är bra – är datasetet obalanserat, det vill säga en klass är mycket vanligare än de andra, kan måttet lura en. Om 95% av datapunkterna tillhör klass 1 och modellen bara gissar klass 1 varje gång får man 95% accuracy, trots att modellen är helt oduglig på att hitta klass 0.

**c) Precision**

$$Precision = \frac{TP}{TP + FP}$$

Precision talar om hur stor andel av de positiva prediktionerna som faktiskt stämde. Hög precision betyder alltså att när modellen säger "positiv" kan man lita på det.

**d) Recall**

$$Recall = \frac{TP}{TP + FN}$$

Recall mäter istället hur stor andel av alla faktiskt positiva fall som modellen lyckades hitta. Kallas ibland TPR (True Positive Rate) eller sensitivity.

**e) F1-score**

$$F_1 = \frac{2}{recall^{-1} + precision^{-1}} = 2\frac{precision \cdot recall}{precision + recall} = \frac{2TP}{2TP + FP + FN}$$

F1-score är det harmoniska medelvärdet av precision och recall. Poängen med att använda just harmoniskt medelvärde är att båda måtten måste vara höga för att F1 ska bli högt – en modell kan inte "gömma" en dålig recall bakom en skyhög precision. Bra mått när man vill ha en balans mellan de två.

**f) ROC-kurvan**

ROC-kurvan (Receiver Operating Characteristic curve) liknar precision-recall-kurvan men visar istället sambandet mellan TPR (recall) och FPR (False Positive Rate):

$$TPR = \frac{TP}{TP+FN} = recall \qquad FPR = \frac{FP}{FP+TN}$$

Genom att flytta på tröskelvärdet (threshold) för när en observation räknas som positiv rör man sig längs kurvan. En perfekt modell går rakt upp till punkten (0, 1) – alla positiva hittas utan ett enda falskt positivt. En modell som bara gissar slumpmässigt ger en rak diagonal från (0,0) till (1,1). Arean under kurvan, ROC-AUC, används som ett sammanfattande mått: 1.0 är perfekt, 0.5 är lika bra som slumpen och under 0.5 är sämre än slumpen. Värt att komma ihåg är att ROC-AUC kan ge en skev bild vid kraftigt obalanserade dataset, eftersom FPR ofta blir lågt ändå även om modellen är dålig på att hitta den positiva klassen.

## Fråga 4 – Vad är precision-recall tradeoff för något?

**Precision-recall tradeoff** (avsnitt 4.2.3) går ut på att en högre recall i regel innebär en lägre precision, och tvärtom – de två måtten drar helt enkelt åt olika håll.

Anledningen är att klassificeringsmodeller ofta bygger på en skattad sannolikhet för att en observation hör till en viss klass, och sen finns det en threshold (som standard 50%) som avgör var gränsen går för att klassa något som positivt. Höjer man den blir modellen mer försiktig och klassar bara de säkraste fallen som positiva, vilket ger färre falska positiva och alltså högre precision, men samtidigt fler falska negativa eftersom fler faktiska positiva fall missas – recall sjunker. Sänker man tröskeln istället klassas fler observationer som positiva, vilket ger högre recall men lägre precision av samma anledning fast tvärtom.

Ett tankeexperiment: en modell som alltid gissar positivt får 100% recall, för den missar aldrig ett enda positivt fall. Men precisionen rasar eftersom en massa faktiskt negativa observationer också klassas som positiva.

Boken tar upp två bra exempel. Vid spamfiltrering ger hög precision att mejl som markeras som spam oftast verkligen är spam, men priset är att fler spam-mejl slinker igenom till inkorgen (lägre recall). Inom sjukvård, där man vill identifiera sjuka patienter, ger hög recall att man hittar fler av de faktiskt sjuka, men då blir det också fler friska som felaktigt klassas som sjuka (lägre precision).

Eftersom man sällan kan maximera båda samtidigt är F1-score ett bra mått att använda när man vill ha en balans mellan precision och recall (se fråga 3e).

## Fråga 5 – Vanligt förekommande klassificeringsmodeller: a) Logistisk regression, b) SVM, c) Beslutsträd, d) Ensemble learning, e) Random forest, f) Extra trees.

**a) Logistisk regression**

Trots namnet används logistisk regression för klassificering, inte regression (avsnitt 4.3.1). Modellen skattar en sannolikhet p för att en datapunkt tillhör en viss klass, och är $p \geq 50\%$ predikteras den positiva klassen (1), annars den negativa (0).

Precis som linjär regression utgår modellen från en linjär kombination av indata: $\theta_0 + \theta_1 x_1 + ... + \theta_p x_p$. Problemet är bara att det uttrycket kan bli vilket tal som helst, medan en sannolikhet måste ligga mellan 0 och 1. Löser man ut p ur log-oddset $\log(p/(1-p))$ landar man i den logistiska funktionen (en sigmoid):

$$p = \sigma(x) = \frac{1}{1 + e^{-x}}$$

Den har den klassiska S-formen och håller sig alltid mellan 0 och 1, vilket gör att den går att tolka som en sannolikhet. Eftersom modellen i grunden är linjär blir beslutsgränserna också linjära (Figur 4.8), vilket gör den sämre på data med olinjär struktur om man inte lägger till polynomvariabler (Figur 4.9). Modellen är i grunden binär, men kan byggas ut till multiklass antingen via OvR/OvO (fråga 2) eller via multinomial logistic regression (softmax regression).

**b) Support vector machines (SVM)**

SVM kan användas både för regression och klassificering (avsnitt 4.3.2). Vid klassificering är målet att hitta en så bred "väg" eller marginal som möjligt mellan klasserna – punkterna som ligger precis på marginalens kant kallas stödvektorer (support vectors), därav namnet. En linjär SVM kan skapas med `LinearSVC` (snabbare, optimerad) eller `SVC(kernel="linear")`. Med olika kernels, till exempel `"poly"`, klarar SVM även data som inte är linjärt separerbar.

**c) Beslutsträd**

Beslutsträd (decision trees, avsnitt 4.3.3) kan liksom SVM hantera både regression och klassificering. Ett träd byggs upp av noder – rotnoden är den första, lövnoderna är de sista. För att klassificera går man från rotnoden, genom de inre noderna, ner till en lövnod som representerar den predikterade klassen. Vid varje förgrening väljer modellen den variabel och det tröskelvärde som ger renast möjliga (mest homogena) noder, mätt med gini-koefficienten:

$$G_i = 1 - \sum_{k=1}^{K} p_{i,k}^2$$

Ett problem är att scikit-learns standardinställningar (till exempel `max_depth=None`) tillåter väldigt djupa träd, så man bör i princip alltid köra grid search för att slippa överanpassning.

**d) Ensemble learning**

Grundidén med ensemble learning (avsnitt 4.3.4) är att slå ihop flera modeller till en som predikterar och generaliserar bättre än vad de enskilda modellerna gör var för sig. Vid klassificering används oftast en voting classifier, och där finns två varianter. Hard voting innebär att varje modell röstar på en klass och majoriteten vinner. Soft voting innebär istället att varje modell ger en sannolikhet per klass, ett genomsnitt räknas ut och klassen med högst genomsnittlig sannolikhet väljs – det kräver dock att alla ingående modeller faktiskt kan skatta sannolikheter.

Andra ensemble-metoder som nämns är bagging (`bootstrap=True`, urval med återläggning) och pasting (urval utan återläggning) via `BaggingClassifier`, samt boosting (till exempel XGBoost), där modellerna tränas i följd så att varje ny modell rättar till föregående modells misstag.

Ett kul räkneexempel från boken: även om varje enskild klassificerare bara är obetydligt bättre än slumpen (säg 51% träffsäkerhet), kan man med binomialfördelningen visa att ett ensemble av 1000 sådana modeller, med hard voting, klarar sig med hela ca 72,6% träffsäkerhet. Ger en bra känsla för varför ensemble learning faktiskt funkar.

**e) Random forest**

Random forest (avsnitt 4.3.5) är ett ensemble av beslutsträd, oftast byggt med bagging. Man skulle kunna bygga det själv med `BaggingClassifier(DecisionTreeClassifier(), ...)`, men scikit-learn har en färdig, optimerad variant i `RandomForestClassifier`. Den stora skillnaden mot ett vanligt bagging-ensemble av träd är att `max_features` som standard är `'sqrt'` – bara ett slumpmässigt urval av features övervägs vid varje förgrening istället för alla, vilket gör de enskilda träden mindre lika varandra och hela modellen mer robust.

**f) Extra trees**

Extra trees (Extremely Randomized Trees, `ExtraTreesClassifier`) drar slumpen ett steg längre än random forest. Utöver ett slumpmässigt urval av features väljs även tröskelvärdet för varje förgrening slumpmässigt, istället för att räkna fram det optimala via gini-koefficienten. Eftersom man slipper den beräkningen tränas modellen generellt snabbare. Extra trees får oftast högre bias men lägre varians än random forest, och kan totalt sett prestera bättre tack vare bias-variance trade-off.

## Fråga 6 – Vad innebär feature importance med hjälp av trädmodeller?

Beslutsträd, random forest och extra trees har alla ett attribut som heter `feature_importances_`, som ger en siffra per variabel för hur viktig den varit för modellens prediktioner. Ju högre siffra, desto viktigare variabel, och alla siffrorna tillsammans summerar till 1.

```python
print(rf_clf.feature_importances_)
print(ert_clf.feature_importances_)
```

```
[0.43042283 0.56957717]
[0.46523419 0.53476581]
```

Det praktiska med det här är att man kan använda `feature_importances_` för variabelselektion – man tittar helt enkelt på vilka variabler som verkar spela störst roll och kan då avgöra vilka som är värda att ha med i en (kanske helt annan) modell. Boken poängterar att man kan göra det här även om man aldrig tänker använda själva trädmodellen för de slutgiltiga prediktionerna. Man tränar alltså en random forest bara för att kika på `feature_importances_` och förstå datan bättre, inte nödvändigtvis för att använda den som slutmodell.

## Fråga 7 – (Resonemangsfråga) Precision vs. recall, och rättsväsendet

Om Stina vill ha så hög precision som möjligt kommer recall generellt att sjunka – det är själva kärnan i **precision-recall tradeoff** (fråga 4). En modell med extremt hög precision klassar bara något som positivt när den är riktigt säker, vilket minskar antalet falska positiva men samtidigt gör att fler faktiska positiva fall missas, alltså fler falska negativa och lägre recall. I ytterlighetsfallet, en modell som aldrig predikterar positivt om den inte är 100% säker, kan man få skyhög precision men väldigt låg recall eftersom så många riktiga positiva fall missas på vägen.

Så när vill man egentligen ha så hög precision som möjligt? Kort svar: när ett falskt positivt kostar mycket. Vid spamfiltrering är ett falskt positivt (ett viktigt mejl hamnar i skräpposten) potentiellt allvarligt om mottagaren aldrig ser mejlet, så där prioriteras hög precision även om något enstaka spam-mejl då slinker igenom. Samma resonemang gäller rekommendationssystem och marknadsföring, där man vill vara säker på att erbjudanden verkligen är relevanta så man inte irriterar kunderna, eller automatiserade beslut med stora konsekvenser för individen, där ett felaktigt positivt beslut är svårt att ångra.

Motsatsen gäller förstås när kostnaden för ett falskt negativt – att missa ett faktiskt positivt fall – är hög. Boken tar upp sjukvårdsexemplet (avsnitt 4.2.3): optimerar man för hög precision där riskerar man att missa patienter som faktiskt är sjuka, vilket kan få allvarliga konsekvenser. Då vill man hellre prioritera recall.

Rättsväsendet är ett bra exempel att koppla ihop det här med. Tänker man sig en fällande dom som en "positiv" prediktion (personen är skyldig) blir ett falskt positivt att en oskyldig person döms felaktigt (typ I-fel), och ett falskt negativt att en skyldig person frias felaktigt (typ II-fel). Eftersom en fängelsedom är så pass allvarlig och svår att ångra för en oskyldig person strävar rättsväsendet efter extremt hög precision – man vill i princip aldrig döma en oskyldig. Det brukar sammanfattas som "hellre fria tio skyldiga än fälla en oskyldig" (jämför Blackstone's ratio och beviskravet "bortom rimligt tvivel"). Konsekvensen, precis enligt precision-recall tradeoff, är att man medvetet accepterar en lägre recall: en del faktiskt skyldiga kommer att frias eftersom bevisningen inte räcker upp till det höga beviskravet. Det är alltså ett medvetet samhälleligt val i tradeoffen – man prioriterar att minimera antalet felaktiga fällande domar, även om det kostar i form av fler skyldiga som går fria.

## Fråga 8 – (Resonemangsfråga) Tolkning av Figur 4.8 på sidan 175

Figur 4.8 visar beslutsgränserna för en tränad logistisk regressionsmodell som klassificerar observationer i en av två klasser (klass 0/lila eller klass 1/gul) utifrån två variabler, x1 och x2. Datan är skapad med `make_moons`, samma halvmåneformade dataset som dyker upp genomgående i avsnitt 4.3.

När jag kollar på figuren ser jag att varje punkt är en observation färgad efter sin sanna klass – lila för klass 0, gul för klass 1. Bakgrundsfärgen visar istället den sannolikhet, mellan 0 och 1, som modellen skattar för att en given punkt i planet tillhör klass 1, och den är graderad från lila (sannolikhet nära 0) via grönt/turkost (ungefär 0.5) till gult (nära 1). Det som sticker ut är att övergångarna mellan färgerna bildar raka, diagonala band. Det är för att logistisk regression är en **linjär modell**, så beslutsgränsen – där sannolikheten är precis 50%, den turkosa linjen mitt i bilden – blir en rät linje, precis som texten säger: "Logistisk regression är en linjär modell, och således är beslutsgränserna också linjära".

Om jag tolkar det hela så har modellen lärt sig en generell trend: högre värden på x1 och x2 (nedre högra delen av figuren) hänger ihop med klass 1, medan lägre värden (övre vänstra delen) hänger ihop med klass 0. Men eftersom `make_moons` har en olinjär, halvmåneformad struktur med ett visst överlapp mellan klasserna kan en enda rät linje aldrig separera dem perfekt. Det syns tydligt i mittenområdet, där bakgrunden är turkos/grön (nära 50% sannolikhet) och lila och gula punkter blandas – de punkterna ligger nära eller på fel sida om den linjära beslutsgränsen och riskerar att bli felklassificerade.

Det här visar egentligen en generell begränsning hos logistisk regression: den funkar bäst när sambandet mellan x och y är ungefär linjärt. På data med tydlig olinjär struktur, som här, kan man antingen lägga till polynomvariabler (`PolynomialFeatures`) för att få olinjära beslutsgränser – vilket är precis vad Figur 4.9 på nästa sida visar, där samma dataset får en böjd beslutsgräns och färre felklassificeringar i mitten – eller byta till en mer flexibel modell som SVM eller random forest.

## Fråga 9 – (Resonemangsfråga) Logiken bakom .fit_transform() vs. .transform()

Citatet kommer från kodexemplet om MNIST-klassificering (avsnitt 4.4.1, sidan 209), där datan standardiseras med `StandardScaler`:

```python
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)
```

Grejen är att `StandardScaler` (och andra transformers) måste lära sig parametrar från datan innan den kan transformera något – i det här fallet handlar det om att räkna ut medelvärde och standardavvikelse för varje variabel. `.fit_transform()` gör två saker på en gång: den lär (fit) sig dessa parametrar utifrån datan den får in, och transformerar sedan samma data utifrån de inlärda parametrarna. `.transform()` gör bara den andra biten – den återanvänder redan inlärda parametrar för att transformera ny data, utan att räkna om dem.

Genom att bara köra `.fit_transform()` på träningsdatan ser man till att medelvärdet och standardavvikelsen – precis som alla andra parametrar en modell lär sig – enbart baseras på träningsdatan. På validerings- och testdatan används sedan bara `.transform()`, som återanvänder exakt samma parametrar som räknades ut på träningsdatan.

Varför spelar det här roll? Om man istället hade kört `.fit_transform()` även på validerings- eller testdatan hade skalningen delvis byggt på information från data som modellen egentligen inte ska ha sett i förväg. Det kallas **data leakage**, och resultatet blir en alltför optimistisk och missvisande bild av hur bra modellen faktiskt presterar på ny, osedd data, eftersom validerings-/testdatan då inte längre är helt oberoende av den bearbetning modellen bygger på. Samma princip som varför man delar upp data i träning/validering/test över huvud taget (jämför kapitel 1): all inlärning, oavsett om det gäller en modells parametrar eller en transformers skalningsparametrar, ska bara ske utifrån träningsdatan – annars får man inte en rättvisande skattning av modellens generaliseringsförmåga.

## Fråga 11 – (Koduppgift) Förklara och tolka `classification_report`

Koden importerar `classification_report` från scikit-learn och bygger en rapport som sammanfattar precision, recall och F1-score (se fråga 3) för varje klass i ett multiklassproblem, givet en lista med sanna värden (`y_true`) och predikterade värden (`y_pred`).

```python
from sklearn.metrics import classification_report

y_true = [0, 1, 2, 2, 2]
y_pred = [0, 0, 2, 2, 1]
target_names = ['class 0', 'class 1', 'class 2']
print(classification_report(y_true, y_pred, target_names=target_names))
```

Det här är samma exempel som boken visar (sidan 163, avsnitt 4.2.4), och nedan är den faktiska outputen, körd i Python:

```
              precision    recall  f1-score   support

     class 0       0.50      1.00      0.67         1
     class 1       0.00      0.00      0.00         1
     class 2       1.00      0.67      0.80         3

    accuracy                           0.60         5
   macro avg       0.50      0.56      0.49         5
weighted avg       0.70      0.60      0.61         5
```

Om jag tolkar outputen rad för rad: class 0 finns sant en gång (support = 1, index 0), och modellen gissar på class 0 två gånger (index 0 och 1). Av de två gissningarna var en rätt, så precision blir 1/2 = 0.50, och eftersom den enda faktiska class 0:an hittades blir recall 1/1 = 1.00. Class 1 finns också sant en gång (index 1), men modellen missar den helt – den gissar 0 istället – och gissar dessutom class 1 en gång där sanningen egentligen var class 2 (index 4). Ingen enda träff, så både precision och recall blir 0.00. Class 2 finns sant tre gånger (index 2, 3, 4), modellen gissar class 2 två gånger (index 2, 3) och båda är rätt, vilket ger precision 2/2 = 1.00, men av de tre faktiska class 2:orna hittades bara två, så recall blir 2/3 ≈ 0.67.

F1-score är som vanligt det harmoniska medelvärdet av precision och recall för respektive klass, och support anger hur många sanna observationer det fanns av varje klass. Accuracy blir andelen totalt korrekta prediktioner, alltså 3/5 = 0.60 (index 0, 2 och 3 är rätt, index 1 och 4 är fel). Macro avg är det enkla, oviktade medelvärdet över de tre klasserna – för precision till exempel (0.50+0.00+1.00)/3 ≈ 0.50 – medan weighted avg viktas efter support, så class 2 (som har flest sanna observationer) väger tyngst i totalen.

Det som är lite klurigt, och som gör rapporten mer användbar än bara accuracy, är att den avslöjar saker en enda siffra döljer. Här ser man tydligt att modellen presterar riktigt dåligt på class 1 (0.00 rakt igenom) trots att den totala accuracyn på 0.60 kan se hyfsad ut vid en snabb blick.

In [ ]:
from sklearn.metrics import classification_report

y_true = [0, 1, 2, 2, 2]
y_pred = [0, 0, 2, 2, 1]
target_names = ['class 0', 'class 1', 'class 2']
print(classification_report(y_true, y_pred, target_names=target_names))

## Fråga 13 – (Koduppgift) Komplett ML-flöde för hr_employee_data.xlsx (y = left)

I föregående kapitel gjordes en EDA på datasetet `hr_employee_data.xlsx`. Nedan byggs ett komplett klassificerings-flöde där den beroende variabeln är **`left`** (1 = anställd har lämnat företaget, 0 = anställd är kvar). Flödet följer samma struktur som kodexemplen i Avsnitt 4.4: inläsning, uppdelning i träning/validering/test, hantering av kategoriska variabler, skalning (`.fit_transform()` på träningsdata, `.transform()` på validering/test – se Fråga 9), träning av en klassificeringsmodell, samt utvärdering med accuracy, precision, recall, F1-score och confusion matrix.

**OBS:** filen `hr_employee_data.xlsx` finns inte lokalt i den här miljön – den behöver hämtas från bokens GitHub-repo innan cellen nedan kan köras. Koden nedan är därför **inte exekverad**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)

# 1. Läs in data
df = pd.read_excel("hr_employee_data.xlsx")

# 2. Separera beroende variabel (y) och oberoende variabler (X)
y = df["left"]
X = df.drop(columns=["left"])

# 3. Hantera kategoriska kolumner (t.ex. "department" och "salary")
#    med one-hot-encoding, precis som beskrivs i Kapitel 1, Fråga 3f.
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
X = pd.get_dummies(X, columns=categorical_cols, drop_first=False)

# 4. Dela upp i träning, validering och test (60-20-20, se Kapitel 1, Fråga 11)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

# 5. Skala numeriska variabler. Notera: fit_transform ENDAST på träningsdata,
#    transform på validering/test (se Fråga 9 - undviker data leakage).
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# 6. Träna en klassificeringsmodell, här en Random Forest (se Fråga 5e)
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# 7. Utvärdera på valideringsdata
y_val_pred = model.predict(X_val)
print("--- Utvärdering på valideringsdata ---")
print("Accuracy:", round(accuracy_score(y_val, y_val_pred), 3))
print("Precision:", round(precision_score(y_val, y_val_pred), 3))
print("Recall:", round(recall_score(y_val, y_val_pred), 3))
print("F1-score:", round(f1_score(y_val, y_val_pred), 3))
print(classification_report(y_val, y_val_pred, target_names=["Stannar (0)", "Lämnar (1)"]))

cm = confusion_matrix(y_val, y_val_pred)
ConfusionMatrixDisplay(cm, display_labels=["Stannar (0)", "Lämnar (1)"]).plot(cmap="Blues")
plt.title("Confusion Matrix - Valideringsdata")
plt.show()

# 8. Slutlig utvärdering på testdata (efter ev. modellval/omträning på train+val)
y_test_pred = model.predict(X_test)
print("--- Utvärdering på testdata ---")
print(classification_report(y_test, y_test_pred, target_names=["Stannar (0)", "Lämnar (1)"]))

## Fråga 15 – (Koduppgift) Predicera egna handskrivna siffror med en tränad MNIST-modell

I avsnitt 4.4.1 tränas en modell (till exempel random forest) på MNIST, där varje bild är 28×28 pixlar i gråskala med en mörk siffra på svart bakgrund, och pixelvärdena är (efter skalning) centrerade och normaliserade. Vill man använda en sådan modell på ett eget foto, säg taget med mobilen på ett papper med en handskriven siffra, räcker det inte att bara mata in bilden rakt av. Kameran ger en helt annan typ av bild – ofta hög upplösning, i färg, med mörk siffra på ljus bakgrund, ojämn belysning, och siffran är varken centrerad eller i rätt storlek.

Huvudjobbet ligger därför i att preprocessa bilden så att den liknar hur en MNIST-bild ser ut, ungefär i den här ordningen:

1. Konvertera bilden till gråskala.
2. Beskär bilden till siffrans bounding box och skala den till 28×28 pixlar, med bevarad proportion och padding, likt MNIST.
3. Invertera färgerna om bakgrunden är ljus, eftersom MNIST har vit/ljus siffra på svart bakgrund.
4. Centrera siffran, till exempel utifrån pixlarnas "tyngdpunkt" (centroid), på samma sätt som MNIST-bilderna är centrerade.
5. Normalisera pixelvärdena och applicera samma `scaler.transform()` som användes på träningsdatan (se fråga 9 – inte en ny `fit`!).
6. Platta ut bilden (`.flatten()`) till en 784-lång vektor och skicka in den i den tränade modellens `.predict()`.

Eftersom det inte finns något riktigt foto tillgängligt i den här miljön är koden nedan inte körd. Värt att nämna är också att verkliga foton ofta skiljer sig ganska mycket från MNIST:s "städade" data – pennstreckets tjocklek, kontrast, skuggor, lutning på siffran och så vidare varierar – vilket gör att en modell tränad enbart på MNIST ofta presterar sämre på riktiga foton än på MNIST:s eget testset. Ett ganska klassiskt exempel på **distributionsskillnad** mellan träningsdata och verklig, osedd data.

In [ ]:
import numpy as np
from PIL import Image, ImageOps

def preprocess_photo_for_mnist(image_path, scaler=None):
    """
    Preprocessar ett foto av en handskriven siffra så att den kan
    predikteras av en modell tränad på MNIST (28x28, gråskala,
    ljus siffra på mörk bakgrund, centrerad, normaliserad).
    """
    # 1. Öppna bilden och konvertera till gråskala
    img = Image.open(image_path).convert("L")

    # 2. Invertera om bakgrunden är ljus (MNIST har vit siffra på svart botten).
    #    Antag att bakgrunden är den vanligaste pixelfärgen i bilden.
    if np.array(img).mean() > 127:
        img = ImageOps.invert(img)

    # 3. Beskär till siffrans bounding box (ta bort tomt utrymme runt siffran)
    arr = np.array(img)
    threshold = 30
    coords = np.column_stack(np.where(arr > threshold))
    if coords.size > 0:
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        img = img.crop((x_min, y_min, x_max + 1, y_max + 1))

    # 4. Skala med bevarad proportion så längsta sidan blir 20 px
    #    (MNIST-siffror upptar ca 20x20 px inuti en 28x28 bild)
    img.thumbnail((20, 20), Image.LANCZOS)

    # 5. Klistra in i en 28x28 svart bakgrund, centrerat
    canvas = Image.new("L", (28, 28), color=0)
    upper_left = ((28 - img.width) // 2, (28 - img.height) // 2)
    canvas.paste(img, upper_left)

    # 6. Konvertera till numpy-array och normalisera/platta ut
    digit_array = np.array(canvas, dtype=np.float64).reshape(1, -1)  # shape (1, 784)

    # 7. Applicera SAMMA scaler som användes vid träning (endast .transform(),
    #    se Fråga 9 - vi ska inte fitta om skalaren på ny data!)
    if scaler is not None:
        digit_array = scaler.transform(digit_array)

    return digit_array


# Exempel på användning (kräver en tränad modell + scaler från Avsnitt 4.4.1,
# samt ett riktigt foto som inte finns tillgängligt i den här miljön):
#
# processed = preprocess_photo_for_mnist("mitt_foto_pa_en_sjua.jpg", scaler=scaler)
# prediction = model.predict(processed)
# print("Modellen predikterar siffran:", prediction[0])